In [1]:
import scipy as sp
import numpy as np
import pandas as pd
import pickle
import os

from tqdm.notebook import tqdm

# pd.set_option('display.max_rows', 50)

In [2]:
def mean_confidence_interval(x, confidence=0.95):
    m, se = np.mean(x), sp.stats.sem(x)
    h = se * sp.stats.t.ppf((1 + confidence) / 2., x.shape[0] - 1)
    return np.round(m - h, 2), np.round(m, 2), np.round(m + h, 2)

In [4]:
folder_path = "./sweep_results/"
files = [folder_path + entry.name for entry in os.scandir(folder_path) if (entry.is_file() and "loc4" in entry.name)]
print(files)

['./sweep_results/sweep_loc4_30_0.001_128.pkl', './sweep_results/sweep_loc4_5_0.0001_128.pkl', './sweep_results/sweep_loc4_20_0.001_32.pkl', './sweep_results/sweep_loc4_20_0.001_128.pkl', './sweep_results/sweep_loc4_20_0.0001_32.pkl', './sweep_results/sweep_loc4_2_0.0001_128.pkl', './sweep_results/sweep_loc4_50_0.001_64.pkl', './sweep_results/sweep_loc4_50_0.001_32.pkl', './sweep_results/sweep_loc4_20_0.0001_64.pkl', './sweep_results/sweep_loc4_50_0.0001_128.pkl', './sweep_results/sweep_loc4_50_0.0001_64.pkl', './sweep_results/sweep_loc4_30_0.001_32.pkl', './sweep_results/sweep_loc4_30_0.0001_128.pkl', './sweep_results/sweep_loc4_10_0.0001_32.pkl', './sweep_results/sweep_loc4_30_0.0001_64.pkl', './sweep_results/sweep_loc4_50_0.0001_32.pkl', './sweep_results/sweep_loc4_10_0.0001_128.pkl', './sweep_results/sweep_loc4_10_0.001_64.pkl', './sweep_results/sweep_loc4_50_0.001_128.pkl', './sweep_results/sweep_loc4_10_0.0001_64.pkl', './sweep_results/sweep_loc4_2_0.0001_32.pkl', './sweep_result

In [6]:
final = pd.DataFrame()
for file in tqdm(files):
    with open(file, "rb") as f:
        results = pickle.load(f)

    keys = [x for x in results.keys() if "mask" not in x and "true" not in x]
    processed_results = {key: {e: [] for e in ["value", "time"]} for key in keys}
    
    for key in keys:
        tmp_mean = np.mean((results[key]["value"] - results["- - true"]["value"][:, :1]) ** 2, axis=0)
        processed_results[key]["value"] = sorted(list(tmp_mean))
        processed_results[key]["time"] = sorted(results[key]["time"])

    tmp = pd.DataFrame(processed_results).transpose()
    tmp.reset_index(inplace=True)
    
    tmp["latent_dim"] = tmp["index"].apply(lambda x: x.split()[0]).astype(int)
    tmp["time_feature_dim"] = tmp["index"].apply(lambda x: x.split()[1]).astype(int)
    tmp["score_type"] = tmp["index"].apply(lambda x: x.split()[2])
    
    tmp["value"] = tmp.value.apply(lambda x: np.round(np.array(x), 2))
    tmp["time"] = tmp.time.apply(lambda x: np.round(np.array(x), 2))
    
    special = file.split("/")[-1][:-4].split("_")[2:]
    tmp["num_dims"] = int(special[0])
    tmp["lr"] = float(special[1])
    tmp["cond_latent_dim"] = int(special[2])
    
    tmp.drop("index", inplace=True, axis=1)
    tmp = tmp[["num_dims", "lr", "time_feature_dim", "latent_dim", "cond_latent_dim", "score_type", "value", "time"]]

    final = pd.concat([final, tmp])
final.sort_values(["num_dims", "lr", "time_feature_dim", "latent_dim", "cond_latent_dim", "score_type"], ascending=True, inplace=True)
final

  0%|          | 0/36 [00:00<?, ?it/s]

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time
0,2,0.0001,1,32,32,naive,"[5.3, 7.03, 8.56]","[59.11, 59.82, 60.48]"
1,2,0.0001,1,32,32,rl,"[1.05, 1.94, 2.3]","[21.26, 21.79, 23.87]"
0,2,0.0001,1,32,64,naive,"[8.63, 10.61, 10.93]","[59.91, 60.08, 60.28]"
1,2,0.0001,1,32,64,rl,"[0.86, 2.63, 3.56]","[20.19, 21.91, 22.57]"
0,2,0.0001,1,32,128,naive,"[6.42, 11.65, 15.53]","[59.68, 63.86, 65.56]"
...,...,...,...,...,...,...,...,...
31,50,0.0010,128,256,64,rl,"[77932.43, 82693.61, 87885.83]","[13.17, 14.55, 23.55]"
38,50,0.0010,128,512,32,naive,"[11663.99, 18082.63, 22835.31]","[33.85, 35.5, 36.84]"
39,50,0.0010,128,512,32,rl,"[80631.29, 83143.34, 83805.75]","[18.7, 21.48, 22.91]"
38,50,0.0010,128,512,64,naive,"[14923.99, 25152.03, 50054.12]","[37.75, 39.53, 41.02]"


In [7]:
final["mean_value"] = final.value.apply(np.mean)
final["var_value"] = final.value.apply(np.std)

/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


In [8]:
final[final.num_dims == 2].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value
5,2,0.0001,64,32,128,rl,"[0.21, 0.57, 0.75]","[22.11, 25.07, 27.16]",0.510000,0.224499
37,2,0.0001,64,512,128,rl,"[0.26, 0.54, 1.0]","[29.47, 29.81, 32.66]",0.600000,0.305068
13,2,0.0001,64,64,128,rl,"[0.16, 0.96, 0.98]","[20.49, 21.8, 25.66]",0.700000,0.381925
7,2,0.0001,128,32,64,rl,"[0.74, 0.81, 0.81]","[24.9, 25.93, 26.4]",0.786667,0.032998
7,2,0.0001,128,32,128,rl,"[0.72, 0.81, 1.1]","[24.89, 25.17, 26.31]",0.876667,0.162138
15,2,0.0001,128,64,32,rl,"[0.55, 0.6, 1.49]","[23.74, 27.31, 29.23]",0.880000,0.431818
19,2,0.0001,32,128,64,rl,"[0.81, 0.87, 0.98]","[21.05, 21.55, 22.67]",0.886667,0.070396
23,2,0.0001,128,128,64,rl,"[0.36, 0.8, 1.57]","[23.69, 23.98, 26.9]",0.910000,0.500067
13,2,0.0001,64,64,64,rl,"[0.72, 0.93, 1.12]","[23.37, 23.56, 23.87]",0.923333,0.163367
29,2,0.0001,64,256,32,rl,"[0.56, 0.63, 1.66]","[23.99, 24.74, 24.93]",0.950000,0.502858


In [9]:
final[final.num_dims == 5].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value
33,5,0.0001,1,512,128,rl,"[2.68, 2.97, 5.52]","[19.49, 19.65, 23.01]",3.723333,1.275940
27,5,0.0001,32,256,32,rl,"[1.08, 2.28, 10.37]","[17.13, 17.25, 18.37]",4.576667,4.125695
35,5,0.0001,32,512,128,rl,"[2.41, 4.09, 7.97]","[20.79, 20.84, 21.96]",4.823333,2.328338
9,5,0.0001,1,64,64,rl,"[2.31, 5.81, 6.55]","[15.97, 17.29, 18.76]",4.890000,1.849180
13,5,0.0001,64,64,32,rl,"[2.46, 4.46, 7.97]","[16.8, 17.77, 19.14]",4.963333,2.277430
3,5,0.0001,32,32,128,rl,"[3.51, 4.34, 7.15]","[16.88, 18.46, 19.45]",5.000000,1.557584
35,5,0.0001,32,512,32,rl,"[2.54, 3.48, 10.24]","[19.1, 20.9, 22.05]",5.420000,3.429791
19,5,0.0001,32,128,32,rl,"[5.03, 5.19, 6.31]","[17.2, 17.59, 19.33]",5.510000,0.569444
22,5,0.0001,128,128,64,naive,"[2.88, 7.64, 8.44]","[58.3, 60.32, 60.73]",6.320000,2.454275
35,5,0.0001,32,512,64,rl,"[3.68, 7.24, 8.84]","[17.83, 22.11, 22.56]",6.586667,2.156623


In [15]:
final[final.num_dims == 10].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value
12,10,0.0001,64,64,64,naive,"[15.87, 24.0, 86.89]","[48.39, 48.52, 50.35]",42.253333,31.736921
38,10,0.0001,128,512,32,naive,"[22.78, 46.06, 76.44]","[57.66, 60.72, 61.91]",48.426667,21.970431
30,10,0.0001,128,256,128,naive,"[48.01, 49.32, 55.88]","[51.14, 53.08, 56.74]",51.070000,3.442974
30,10,0.0001,128,256,32,naive,"[33.77, 41.41, 79.57]","[50.83, 51.15, 53.2]",51.583333,20.033847
20,10,0.0001,64,128,32,naive,"[50.31, 53.67, 67.84]","[48.78, 50.24, 51.76]",57.273333,7.596632
18,10,0.0001,32,128,128,naive,"[25.86, 39.79, 115.7]","[49.18, 50.05, 51.15]",60.450000,39.479388
38,10,0.0001,128,512,64,naive,"[7.95, 53.18, 132.9]","[55.78, 60.82, 62.35]",64.676667,51.654336
38,10,0.0001,128,512,128,naive,"[22.71, 62.3, 109.41]","[58.26, 58.54, 60.19]",64.806667,35.439479
14,10,0.0001,128,64,128,naive,"[52.12, 62.22, 94.06]","[50.43, 53.06, 54.16]",69.466667,17.872260
0,10,0.0001,1,32,32,naive,"[48.22, 71.36, 89.75]","[46.7, 47.49, 49.69]",69.776667,16.991477


In [17]:
final[final.num_dims == 20].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value
28,20,0.0010,64,256,32,naive,"[121.18, 199.4, 276.69]","[42.94, 43.34, 44.02]",199.090000,63.487070
34,20,0.0010,32,512,32,naive,"[130.82, 144.65, 459.86]","[48.51, 48.58, 49.13]",245.110000,151.956110
38,20,0.0001,128,512,128,naive,"[206.44, 373.87, 451.76]","[46.79, 49.34, 50.63]",344.023333,102.351008
22,20,0.0001,128,128,32,naive,"[158.09, 369.03, 609.59]","[39.24, 39.75, 42.74]",378.903333,184.456272
38,20,0.0001,128,512,32,naive,"[254.07, 403.62, 614.35]","[46.85, 48.84, 49.79]",424.013333,147.788894
38,20,0.0001,128,512,64,naive,"[194.15, 400.99, 704.92]","[49.03, 49.18, 51.94]",433.353333,209.772952
36,20,0.0001,64,512,128,naive,"[303.03, 569.43, 733.69]","[45.43, 48.51, 49.7]",535.383333,177.456831
30,20,0.0001,128,256,32,naive,"[318.06, 398.24, 899.45]","[40.22, 42.15, 44.04]",538.583333,257.262216
26,20,0.0010,32,256,64,naive,"[351.71, 658.57, 661.23]","[40.87, 41.69, 44.28]",557.170000,145.286218
22,20,0.0001,128,128,64,naive,"[497.55, 552.23, 774.93]","[40.31, 40.94, 41.4]",608.236667,119.965206


In [12]:
final[final.num_dims == 30].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value
38,30,0.0001,128,512,32,naive,"[667.71, 1353.33, 1602.41]","[41.72, 42.34, 42.81]",1207.816667,395.218606
8,30,0.0010,1,64,128,naive,"[710.67, 738.95, 2698.54]","[36.54, 36.66, 38.89]",1382.720000,930.496872
36,30,0.0001,64,512,64,naive,"[832.08, 1764.27, 1997.97]","[41.15, 43.04, 43.87]",1531.440000,503.641559
16,30,0.0010,1,128,64,naive,"[195.62, 2198.7, 2400.26]","[35.4, 36.07, 38.27]",1598.193333,995.176894
16,30,0.0010,1,128,32,naive,"[129.11, 2038.92, 3285.59]","[32.23, 34.54, 36.41]",1817.873333,1298.072345
8,30,0.0010,1,64,64,naive,"[1039.5, 1075.21, 3520.57]","[32.42, 34.02, 35.95]",1878.426667,1161.262200
34,30,0.0001,32,512,128,naive,"[1186.32, 1995.5, 2652.9]","[39.46, 39.67, 44.46]",1944.906667,599.796626
36,30,0.0001,64,512,32,naive,"[1542.55, 2055.33, 2660.74]","[39.74, 40.82, 43.95]",2086.206667,457.020966
34,30,0.0001,32,512,64,naive,"[1492.41, 2456.64, 2718.86]","[42.05, 42.8, 42.85]",2222.636667,527.328532
24,30,0.0010,1,256,32,naive,"[1215.25, 2700.9, 2858.69]","[36.5, 37.49, 39.83]",2258.280000,740.341411


In [24]:
tmp = final[final.num_dims == 2].sort_values("mean_value").head(30)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_2 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 5].sort_values("mean_value").head(30)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_5 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 10].sort_values("mean_value").head(30)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_10 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 20].sort_values("mean_value").head(30)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_20 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 30].sort_values("mean_value").head(30)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_30 = set(map(tuple, list(tmp.values)))

In [25]:
set_2.intersection(set_5).intersection(set_10).intersection(set_20).intersection(set_30)

{(np.int64(1), np.int64(128), np.int64(32)),
 (np.int64(32), np.int64(256), np.int64(32))}

In [34]:
final[final.num_dims == 20].sort_values("mean_value").head(30)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value
11,20,0.0001,32,64,32,rl,"[2.16, 3.64, 5.15]","[12.81, 13.97, 14.45]",3.650000,1.220683
11,20,0.0001,32,64,64,rl,"[2.54, 3.77, 6.12]","[13.87, 14.83, 14.99]",4.143333,1.485179
27,20,0.0001,32,256,32,rl,"[3.76, 4.01, 5.89]","[14.64, 14.73, 15.15]",4.553333,0.950661
17,20,0.0001,1,128,32,rl,"[2.56, 5.44, 6.91]","[13.15, 13.34, 15.19]",4.970000,1.806710
17,20,0.0001,1,128,64,rl,"[2.85, 4.45, 7.64]","[15.09, 15.34, 15.58]",4.980000,1.991097
9,20,0.0001,1,64,64,rl,"[3.41, 5.85, 7.66]","[14.15, 14.36, 14.53]",5.640000,1.741398
27,20,0.0001,32,256,64,rl,"[3.93, 6.27, 8.23]","[16.94, 17.93, 18.24]",6.143333,1.757751
17,20,0.0001,1,128,128,rl,"[6.08, 6.34, 6.74]","[13.94, 15.92, 16.66]",6.386667,0.271457
9,20,0.0001,1,64,128,rl,"[4.51, 4.88, 10.22]","[14.8, 15.16, 16.18]",6.536667,2.608887
37,20,0.0001,64,512,128,rl,"[4.27, 7.95, 8.16]","[16.74, 16.87, 20.69]",6.793333,1.786325
